# CarWorth — Data Cleaning & Feature Engineering
**Input:** `data/raw/vehicles.csv`  
**Output:** `data/processed/vehicles_clean.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

## 1. Load Raw Data

In [ ]:
df = pd.read_csv('../data/raw/vehicles.csv')
print(f'Raw shape: {df.shape}')
df.head(3)

## 2. Drop Irrelevant / High-Missing Columns

In [ ]:
# Drop cols with >60% missing or not useful for ML
drop_cols = [
    'id', 'url', 'region_url', 'image_url', 'description',
    'county',   # >99% missing
    'size',     # ~72% missing
    'VIN',      # identifier, not predictive
    'region',   # redundant with state
]
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
print(f'After dropping cols: {df.shape}')

## 3. Filter Price Outliers

In [ ]:
before = len(df)
df = df[(df['price'] >= 500) & (df['price'] <= 150_000)]
print(f'Removed {before - len(df):,} rows with price outside [$500, $150K]. Remaining: {len(df):,}')

## 4. Filter Year Outliers

In [ ]:
before = len(df)
df = df[(df['year'] >= 1990) & (df['year'] <= 2024)]
print(f'Removed {before - len(df):,} rows with invalid year. Remaining: {len(df):,}')

## 5. Odometer Cleaning

In [ ]:
before = len(df)
df = df[(df['odometer'] >= 0) & (df['odometer'] <= 350_000)]
print(f'Removed {before - len(df):,} rows with invalid odometer. Remaining: {len(df):,}')

## 6. Handle Missing Values

In [ ]:
# Categorical: fill with 'unknown'
cat_fill = ['manufacturer', 'model', 'condition', 'cylinders', 'fuel',
            'title_status', 'transmission', 'drive', 'type', 'paint_color', 'state']

for col in cat_fill:
    if col in df.columns:
        df[col] = df[col].fillna('unknown')

# Numeric: fill with median
df['odometer'] = df['odometer'].fillna(df['odometer'].median())

print('Missing values after fill:')
print(df.isnull().sum()[df.isnull().sum() > 0])

## 7. Feature Engineering

In [ ]:
# Car age
df['car_age'] = 2024 - df['year'].astype(int)

# Age-odometer interaction (average miles per year)
df['miles_per_year'] = df['odometer'] / df['car_age'].replace(0, 1)

# Log price (target for modeling)
df['log_price'] = np.log1p(df['price'])

# Log odometer
df['log_odometer'] = np.log1p(df['odometer'])

# Is luxury brand
luxury_brands = {'bmw', 'mercedes-benz', 'audi', 'lexus', 'cadillac', 'lincoln', 'infiniti', 'acura', 'volvo', 'jaguar', 'land rover', 'porsche', 'tesla'}
df['is_luxury'] = df['manufacturer'].str.lower().isin(luxury_brands).astype(int)

# Is clean title
df['is_clean_title'] = (df['title_status'] == 'clean').astype(int)

# Is automatic transmission
df['is_automatic'] = (df['transmission'] == 'automatic').astype(int)

print('New features added: car_age, miles_per_year, log_price, log_odometer, is_luxury, is_clean_title, is_automatic')
df[['car_age', 'miles_per_year', 'log_price', 'is_luxury']].head()

## 8. Normalize / Clean String Columns

In [ ]:
str_cols = ['manufacturer', 'model', 'fuel', 'transmission', 'drive', 'type', 'condition', 'state', 'paint_color']
for col in str_cols:
    if col in df.columns:
        df[col] = df[col].str.lower().str.strip()

# Map condition to numeric ordinal
condition_map = {'salvage': 0, 'fair': 1, 'good': 2, 'excellent': 3, 'like new': 4, 'new': 5, 'unknown': 2}
df['condition_num'] = df['condition'].map(condition_map).fillna(2)

## 9. Keep Relevant Columns for Modeling

In [ ]:
MODEL_COLS = [
    'price', 'log_price',
    'year', 'car_age', 'odometer', 'log_odometer', 'miles_per_year',
    'manufacturer', 'fuel', 'transmission', 'drive', 'type',
    'condition', 'condition_num', 'cylinders',
    'state', 'paint_color',
    'is_luxury', 'is_clean_title', 'is_automatic'
]

df_model = df[[c for c in MODEL_COLS if c in df.columns]].copy()
print(f'Final shape for modeling: {df_model.shape}')
df_model.dtypes

## 10. Visualize Cleaned Data

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df_model['log_price'].plot(kind='hist', bins=60, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('log(Price) — Clean')

df_model['car_age'].plot(kind='hist', bins=35, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Car Age Distribution')

df_model['log_odometer'].plot(kind='hist', bins=50, ax=axes[2], color='teal', edgecolor='white')
axes[2].set_title('log(Odometer) Distribution')

plt.tight_layout()
plt.savefig('../assets/02_cleaning/clean_distributions.png', dpi=150)
plt.show()

## 11. Save Cleaned Dataset

In [ ]:
df_model.to_csv('../data/processed/vehicles_clean.csv', index=False)
print(f'Saved to data/processed/vehicles_clean.csv — {df_model.shape[0]:,} rows, {df_model.shape[1]} columns')